# Portfolio edition
This is a source-only publishing copy of the original coursework notebook. Saved outputs and account-specific notebook metadata were cleared. Run the notebook with the documented data and compute requirements; historical results are not represented as a fresh execution.


# 42028: Deep Learning and Convolutional Neural Networks
## Assignment 1

| | |
|---|---|
| **Subject** | 42028 — Deep Learning and CNNs |
| **Dataset** | Handwritten Mathematical Symbols — 10 classes, 500 images each |

---

## What This Notebook Does

In this notebook, I classify handwritten mathematical symbols using two classifiers — **SVM** and **ANN** — with four different features: **Raw Pixels**, **LBP**, **HOG**, and **Sobel Edge**.

For each classifier and feature combination, I run **3 different hyperparameter settings** to understand how the choice of hyperparameter affects the classification accuracy. The goal is to find the best performing configuration.

All experiments are tracked using **ClearML**, which logs the hyperparameters, accuracy, confusion matrix, and training curves for every experiment.

---

## Notebook Structure

| Section | Description |
|---|---|
| Section 1 | Install required libraries |
| Section 2 | ClearML setup and credentials |
| Section 3 | Import all libraries |
| Section 4 | Configuration — dataset path, seed, split ratio |
| Section 5 | ClearML data preparation task — log dataset stats |
| Section 6 | Load all images from the dataset folder |
| Section 7 | Train / test split — 70% train, 30% test |
| Section 8 | Feature extraction — Raw Pixels, LBP, HOG, Sobel Edge |
| Section 9 | SVM experiments — 3 values of C per feature |
| Section 10 | ANN experiments — 3 hidden layer sizes per feature |
| Section 11 | Results — comparison table, best confusion matrix, sample predictions |
| Section 12 | ClearML results summary task |

---

## Hyperparameter Settings

**SVM — varying C (regularisation parameter):**

| Feature | Kernel | C values tested | Why this kernel |
|---|---|---|---|
| Raw Pixels | linear | 0.01, 0.1, 1.0 | 2025-dimensional input — linear kernel scales better |
| LBP | rbf | 1.0, 10.0, 100.0 | 26-dimensional histogram — rbf kernel suits low-dimensional features |
| HOG | rbf | 10.0, 50.0, 100.0 | rbf kernel works well with HOG features |
| Sobel Edge | linear | 0.01, 0.1, 1.0 | 2025-dimensional input — same as Raw Pixels |

**ANN — varying hidden layer size:**

| Architecture | Hidden Layer 1 | Hidden Layer 2 |
|---|---|---|
| Small | 128 | 64 |
| Medium | 256 | 128 |
| Large | 512 | 256 |

## Section 1: Install Dependencies

I install all the Python libraries needed for this notebook:
- `opencv-python-headless` — for reading and processing images
- `scikit-image` — for LBP and HOG feature extraction
- `scikit-learn` — for SVM classifier and evaluation metrics
- `matplotlib` and `seaborn` — for plotting graphs and confusion matrices
- `clearml` — for logging and tracking all experiments
- `tensorflow` — for building the ANN using Dense layers

In [ ]:
!pip install opencv-python-headless scikit-image scikit-learn matplotlib seaborn clearml -q
!pip install tensorflow -q
print('All libraries installed successfully ✅')

## Section 2: ClearML Setup

ClearML is used throughout this notebook to track every experiment. For each SVM and ANN run, ClearML logs:
- All hyperparameter settings (kernel, C value, number of neurons, epochs etc.)
- Train and test accuracy
- Confusion matrix image
- Training accuracy and loss curves (for ANN)

I set the API credentials using environment variables. I use `reuse_last_task_id=False` in every `Task.init()` call so that every time this notebook is run, a completely new set of tasks is created in ClearML and all results are updated.

In [ ]:
# ClearML uses your own environment variables; no credentials are bundled.
import os
os.environ.setdefault("CLEARML_WEB_HOST", "https://app.clear.ml/")
os.environ.setdefault("CLEARML_API_HOST", "https://api.clear.ml")
os.environ.setdefault("CLEARML_FILES_HOST", "https://files.clear.ml")
if not (os.environ.get("CLEARML_API_ACCESS_KEY") and os.environ.get("CLEARML_API_SECRET_KEY")):
    os.environ.setdefault("CLEARML_OFFLINE_MODE", "1")
print("ClearML: offline tracking unless your own credentials are configured.")


## Section 3: Import Libraries

I import all the libraries needed for this notebook:

| Library | Used For |
|---|---|
| `cv2` | Reading images in grayscale |
| `numpy` | Array operations and matrix handling |
| `matplotlib`, `seaborn` | Plotting charts and confusion matrices |
| `skimage.feature` | LBP and HOG feature extraction |
| `sklearn.svm` | SVM classifier |
| `sklearn.model_selection` | Train/test split |
| `sklearn.metrics` | Accuracy, confusion matrix, classification report |
| `tensorflow`, `keras` | Building ANN with Dense layers only |
| `clearml` | Logging and tracking experiments |

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import tempfile
%matplotlib inline

from skimage import feature

from sklearn import svm, metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, confusion_matrix,
                              classification_report)

import tensorflow as tf
from tensorflow import keras

from clearml import Task

print(f'All imports successful ✅')
print(f'TensorFlow version : {tf.__version__}')
print(f'OpenCV version     : {cv2.__version__}')

## Section 4: Configuration

I define all the important settings in one place so they are easy to find and change if needed.

Key settings:
- `DATASET_PATH` — the folder containing the dataset. Update this if the folder is in a different location.
- `RANDOM_SEED = 42` — a fixed seed used in `train_test_split` and `svm.SVC`. This ensures the same results every time the notebook is run.
- `TEST_SIZE = 0.30` — 30% of the data is used for testing, 70% for training.
- `CLASS_NAMES` — read directly from the dataset folder. This avoids hardcoding and works for any set of classes.
- `CLEARML_PROJECT` — the project name used in all ClearML tasks so that all experiments appear in the same project.

In [ ]:
DATASET_PATH = os.environ.get('CV_SYMBOLS_DATA', './data/handwritten-symbols')

RANDOM_SEED = 42
TEST_SIZE   = 0.30

CLASS_NAMES  = sorted([d for d in os.listdir(DATASET_PATH)
                        if os.path.isdir(os.path.join(DATASET_PATH, d))])
label_to_idx = {name: idx for idx, name in enumerate(CLASS_NAMES)}
idx_to_label = {idx: name for name, idx in label_to_idx.items()}
NUM_CLASSES  = len(CLASS_NAMES)

CLEARML_PROJECT = '42028-Assignment1-14679308'

print(f'Dataset path     : {DATASET_PATH}')
print(f'Number of classes: {NUM_CLASSES}')
print(f'Class names      : {CLASS_NAMES}')
print(f'Train/Test split : {int((1-TEST_SIZE)*100)}% / {int(TEST_SIZE*100)}%')
print(f'Random seed      : {RANDOM_SEED}')

## Section 5: ClearML — Data Preparation Task

Before running any experiments, I log information about the dataset to ClearML. This creates a data preparation task that records:
- The number of images in each class as scalar values
- A bar chart showing the class distribution
- Sample images from each class (2 per class)

This is useful because it provides a record of what the dataset looks like before any training is done, and allows the examiner to verify the dataset used.

In [ ]:
data_prep_task = Task.init(
    project_name=CLEARML_PROJECT,
    task_name='Data-Preparation-Sweep',
    task_type=Task.TaskTypes.data_processing,
    reuse_last_task_id=False
)

data_prep_task.connect({
    'dataset_path' : DATASET_PATH,
    'num_classes'  : NUM_CLASSES,
    'class_names'  : str(CLASS_NAMES),
    'image_size'   : '45x45',
    'image_mode'   : 'grayscale',
    'train_ratio'  : 1 - TEST_SIZE,
    'test_ratio'   : TEST_SIZE,
    'random_seed'  : RANDOM_SEED,
})

logger_prep = data_prep_task.get_logger()

class_counts = {}
for cls in CLASS_NAMES:
    cls_path = os.path.join(DATASET_PATH, cls)
    count = len([f for f in os.listdir(cls_path) if f.endswith('.jpg')])
    class_counts[cls] = count
    logger_prep.report_scalar(title='Class Image Counts', series=cls,
                               value=count, iteration=0)

print('Class image counts:')
for cls, count in class_counts.items():
    print(f'  {cls:12s}: {count} images')
print(f'  Total       : {sum(class_counts.values())} images')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(list(class_counts.keys()), list(class_counts.values()),
               color='steelblue', edgecolor='black')
ax.set_title('Dataset — Class Distribution (Images per Class)', fontsize=14)
ax.set_xlabel('Class Name', fontsize=12)
ax.set_ylabel('Number of Images', fontsize=12)
ax.set_ylim(0, 600)
for bar, count in zip(bars, class_counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             str(count), ha='center', fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('/tmp/class_distribution.png', dpi=120)
plt.show()

logger_prep.report_image(title='Class Distribution', series='bar chart',
                          local_path='/tmp/class_distribution.png', iteration=0)
print('Class distribution chart logged to ClearML ✅')

In [ ]:
SAMPLES_PER_CLASS = 2
n_cols = SAMPLES_PER_CLASS
n_rows = NUM_CLASSES

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6, n_rows * 2))
fig.suptitle('Sample Images — 2 per Class', fontsize=14,
              fontweight='bold', y=1.01)

for row_idx, cls in enumerate(CLASS_NAMES):
    cls_path  = os.path.join(DATASET_PATH, cls)
    img_files = sorted(os.listdir(cls_path))[:SAMPLES_PER_CLASS]
    for col_idx, fname in enumerate(img_files):
        img = cv2.imread(os.path.join(cls_path, fname), cv2.IMREAD_GRAYSCALE)
        ax  = axes[row_idx][col_idx]
        ax.imshow(img, cmap='gray')
        ax.axis('off')
        if col_idx == 0:
            ax.set_title(f'Class: {cls}', fontsize=9,
                          fontweight='bold', loc='left', pad=3)

plt.tight_layout()
plt.savefig('/tmp/sample_images.png', dpi=120, bbox_inches='tight')
plt.show()

logger_prep.report_image(title='Sample Images per Class', series='grid',
                          local_path='/tmp/sample_images.png', iteration=0)
logger_prep.flush()
data_prep_task.close()
print('Data preparation task closed ✅')

## Section 6: Dataset Loading

In this section, I load all 5000 images from the dataset folder into memory.

How it works:
- I loop through each class folder in the dataset directory
- For each image file, I use `cv2.imread` with `cv2.IMREAD_GRAYSCALE` to read it as a single-channel grayscale image
- Each image is stored as a 45×45 numpy array
- The corresponding class label (as an integer) is stored in a separate array

After loading, `all_images` has shape `(5000, 45, 45)` and `all_labels` has shape `(5000,)`.

In [ ]:
print('Loading dataset...')

all_images = []
all_labels = []

for cls_name in CLASS_NAMES:
    cls_folder = os.path.join(DATASET_PATH, cls_name)
    cls_label  = label_to_idx[cls_name]
    img_files  = [f for f in os.listdir(cls_folder) if f.endswith('.jpg')]

    for fname in img_files:
        img_path = os.path.join(cls_folder, fname)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            all_images.append(img)
            all_labels.append(cls_label)

all_images = np.array(all_images, dtype=np.float32)
all_labels = np.array(all_labels, dtype=np.int32)

print(f'Total images loaded : {len(all_images)}')
print(f'Image array shape   : {all_images.shape}')
print(f'Label array shape   : {all_labels.shape}')
print(f'Pixel value range   : [{all_images.min():.0f}, {all_images.max():.0f}]')
print(f'Classes             : {[idx_to_label[i] for i in range(NUM_CLASSES)]}')

## Section 7: Train / Test Split

I split the dataset into a training set and a test set using `train_test_split` from scikit-learn.

**Important:** This split is performed **only once** and the **same training and test sets are used for every single experiment** in this notebook — all 24 SVM and ANN runs. This ensures a fair and consistent comparison between classifiers and features.

Split settings:
- `test_size = 0.30` → 1500 images for testing, 3500 for training
- `random_state = 42` → ensures the same split is produced every time

After the split:
- `X_train_imgs` — training images, shape `(3500, 45, 45)`
- `X_test_imgs` — test images, shape `(1500, 45, 45)`
- `y_train` — training labels, shape `(3500,)`
- `y_test` — test labels, shape `(1500,)`

In [ ]:
X_train_imgs, X_test_imgs, y_train, y_test = train_test_split(
    all_images,
    all_labels,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED
)

print('Dataset split complete ✅')
print(f'Training samples : {len(X_train_imgs)}  ({int((1-TEST_SIZE)*100)}%)')
print(f'Test samples     : {len(X_test_imgs)}   ({int(TEST_SIZE*100)}%)')
print(f'Training shape   : {X_train_imgs.shape}')
print(f'Test shape       : {X_test_imgs.shape}')
print(f'\nClass distribution in training set:')
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {idx_to_label[u]:12s} ({u}): {c} samples')

## Section 8: Feature Extraction

In this section, I extract four different types of features from the training and test images.

Each feature represents the image in a different way:
- **Raw Pixels** — the original pixel values, no transformation
- **LBP** — encodes local texture patterns using binary comparisons between neighbouring pixels
- **HOG** — captures the distribution of edge directions in local image regions
- **Sobel Edge** — highlights the edges by computing image gradients

All features are extracted once here and reused across all SVM and ANN experiments. This ensures consistency — the same feature values are used for every classifier that trains on that feature.

| Feature | Method | Output Size |
|---|---|---|
| Raw Pixels | Flatten + normalise | 2025 |
| LBP | Histogram of binary patterns | 26 |
| HOG | Gradient orientation histogram | Depends on params |
| Sobel Edge | Combined edge map, flatten | 1849 (43×43 after convolution) |

### Section 8.1: Raw Pixel Features

The raw pixel feature is the simplest way to represent an image.

Steps:
1. Each 45×45 grayscale image is reshaped into a flat 1D vector using `.reshape(-1)`
2. The pixel values (originally 0–255) are divided by 255 to normalise them to the range `[0, 1]`

This gives a **2025-dimensional feature vector** for each image.

Raw pixels preserve all the original information in the image but do not filter out noise or focus on any particular visual property. This makes it a good baseline to compare against more sophisticated features like HOG and LBP.

In [ ]:
X_train_raw = X_train_imgs.reshape(len(X_train_imgs), -1) / 255.0
X_test_raw  = X_test_imgs.reshape(len(X_test_imgs), -1)  / 255.0

print(f'Raw pixel feature extraction complete ✅')
print(f'Training feature shape : {X_train_raw.shape}  (samples x pixels)')
print(f'Test feature shape     : {X_test_raw.shape}')
print(f'Feature value range    : [{X_train_raw.min():.2f}, {X_train_raw.max():.2f}]')

### Section 8.2: LBP (Local Binary Pattern) Features

Local Binary Pattern (LBP) is a texture descriptor that captures local patterns in an image.

How it works:
1. For each pixel, the surrounding neighbours at a given radius are compared to the centre pixel
2. Each comparison produces a 0 or 1, forming a binary pattern
3. A histogram of all the patterns in the image is computed and normalised

Parameters used:
- `numPoints = 24` — number of sampling points around each pixel
- `radius = 8` — the radius of the circle of sampling points

This gives a **26-dimensional feature vector** for each image. LBP works well for texture-based classification but may struggle with symbols that have similar textures but different structures.

In [ ]:
class LocalBinaryPatterns:
    def __init__(self, numPoints, radius):
        self.numPoints = numPoints
        self.radius    = radius

    def LBPfeatures(self, image, eps=1e-7):
        lbp = feature.local_binary_pattern(image, self.numPoints,
                                            self.radius, method='uniform')
        (hist, _) = np.histogram(lbp.ravel(),
                                  bins=np.arange(0, self.numPoints + 3),
                                  range=(0, self.numPoints + 2))
        hist = hist.astype('float')
        hist /= (hist.sum() + eps)
        return hist


LBP_NUM_POINTS = 24
LBP_RADIUS     = 8
lbp_desc = LocalBinaryPatterns(LBP_NUM_POINTS, LBP_RADIUS)

print(f'Extracting LBP features (numPoints={LBP_NUM_POINTS}, radius={LBP_RADIUS})...')

X_train_lbp = []
for img_index in range(len(X_train_imgs)):
    image = X_train_imgs[img_index].astype(np.uint8)
    hist  = lbp_desc.LBPfeatures(image)
    X_train_lbp.append(hist)

X_test_lbp = []
for img_index in range(len(X_test_imgs)):
    image = X_test_imgs[img_index].astype(np.uint8)
    hist  = lbp_desc.LBPfeatures(image)
    X_test_lbp.append(hist)

X_train_lbp = np.array(X_train_lbp)
X_test_lbp  = np.array(X_test_lbp)

print(f'LBP feature extraction complete ✅')
print(f'Training feature shape : {X_train_lbp.shape}  (samples x histogram bins)')
print(f'Test feature shape     : {X_test_lbp.shape}')

### Section 8.3: HOG (Histogram of Oriented Gradients) Features

HOG describes the local appearance and shape of an object by capturing the distribution of gradient orientations.

How it works:
1. The image is divided into small cells
2. In each cell, a histogram of gradient directions is computed
3. Cells are grouped into blocks and the histograms are normalised across each block

Parameters used:
- `orientations = 9` — number of gradient direction bins
- `pixels_per_cell = (8, 8)` — size of each cell in pixels
- `cells_per_block = (2, 2)` — number of cells per normalisation block
- `transform_sqrt = True` — reduces the effect of shadows
- `block_norm = 'L2-Hys'` — normalisation method

HOG is particularly effective for classifying symbols because it captures the stroke direction and shape of each character.

In [ ]:
HOG_ORIENTATIONS    = 9
HOG_PIXELS_PER_CELL = (8, 8)
HOG_CELLS_PER_BLOCK = (2, 2)

print(f'Extracting HOG features (orientations={HOG_ORIENTATIONS}, '
      f'pixels_per_cell={HOG_PIXELS_PER_CELL}, cells_per_block={HOG_CELLS_PER_BLOCK})...')

X_train_hog = []
for img_index in range(len(X_train_imgs)):
    image = X_train_imgs[img_index].astype(np.uint8)
    hog_feat = feature.hog(image,
                            orientations=HOG_ORIENTATIONS,
                            pixels_per_cell=HOG_PIXELS_PER_CELL,
                            cells_per_block=HOG_CELLS_PER_BLOCK,
                            transform_sqrt=True,
                            block_norm='L2-Hys')
    X_train_hog.append(hog_feat)

X_test_hog = []
for img_index in range(len(X_test_imgs)):
    image = X_test_imgs[img_index].astype(np.uint8)
    hog_feat = feature.hog(image,
                            orientations=HOG_ORIENTATIONS,
                            pixels_per_cell=HOG_PIXELS_PER_CELL,
                            cells_per_block=HOG_CELLS_PER_BLOCK,
                            transform_sqrt=True,
                            block_norm='L2-Hys')
    X_test_hog.append(hog_feat)

X_train_hog = np.array(X_train_hog)
X_test_hog  = np.array(X_test_hog)

print(f'HOG feature extraction complete ✅')
print(f'Training feature shape : {X_train_hog.shape}  (samples x hog features)')
print(f'Test feature shape     : {X_test_hog.shape}')

### Section 8.4: Sobel Edge Features

Sobel edge detection highlights the edges in an image by computing the image gradient in the horizontal and vertical directions.

How it works:
1. A horizontal Sobel filter is convolved over the image to detect horizontal edges
2. A vertical Sobel filter is convolved over the image to detect vertical edges
3. The two edge maps are combined by adding their absolute values
4. The combined edge map is normalised to `[0, 1]` and flattened into a 1D feature vector

Filters used:
- `H_filter` — detects horizontal edges
- `V_filter` — detects vertical edges

Sobel features are useful because mathematical symbols are defined by their edges and strokes rather than their texture or raw pixel values.

In [ ]:
def single_step_convolution(slice, filter):
    s    = slice * filter
    cval = np.sum(s)
    return cval


def detect_edge(image, filter):
    (image_row, image_col) = np.shape(image)
    (filter_row, filter_col) = np.shape(filter)
    stride  = 1
    padding = 0
    Ht = image_row - filter_row + 1
    Wd = image_col - filter_row + 1
    detection_result = np.zeros(shape=(Ht, Wd))
    for H in range(Ht):
        for W in range(Wd):
            row_start = H * stride
            row_end   = row_start + filter_row
            col_start = W * stride
            col_end   = col_start + filter_row
            slice_img = image[row_start:row_end, col_start:col_end]
            detection_result[H, W] = single_step_convolution(slice_img, filter)
    return detection_result


H_filter = np.asarray([[-1, -2, -1], [ 0,  0,  0], [ 1,  2,  1]])
V_filter = np.asarray([[-1,  0,  1], [-2,  0,  2], [-1,  0,  1]])


def extract_sobel_features(image):
    img     = image.astype(np.float32)
    h_edges = detect_edge(img, H_filter)
    gX      = np.abs(h_edges).astype(np.float32)
    v_edges = detect_edge(img, V_filter)
    gY      = np.abs(v_edges).astype(np.float32)
    gXY     = gX + gY
    max_val = gXY.max()
    if max_val > 0:
        gXY = gXY / max_val
    return gXY.flatten()


print('Extracting Sobel edge features using detect_edge()...')

X_train_sobel = []
for img_index in range(len(X_train_imgs)):
    image      = X_train_imgs[img_index].astype(np.uint8)
    sobel_feat = extract_sobel_features(image)
    X_train_sobel.append(sobel_feat)

X_test_sobel = []
for img_index in range(len(X_test_imgs)):
    image      = X_test_imgs[img_index].astype(np.uint8)
    sobel_feat = extract_sobel_features(image)
    X_test_sobel.append(sobel_feat)

X_train_sobel = np.array(X_train_sobel)
X_test_sobel  = np.array(X_test_sobel)

print(f'Sobel edge feature extraction complete ✅')
print(f'Training feature shape : {X_train_sobel.shape}')
print(f'Test feature shape     : {X_test_sobel.shape}')

In [ ]:
print('Feature Extraction Summary')
print('=' * 50)
print(f'{'Feature':<15} {'Train Shape':<25} {'Test Shape'}')
print('-' * 50)
print(f'{'Raw Pixels':<15} {str(X_train_raw.shape):<25} {X_test_raw.shape}')
print(f'{'LBP':<15} {str(X_train_lbp.shape):<25} {X_test_lbp.shape}')
print(f'{'HOG':<15} {str(X_train_hog.shape):<25} {X_test_hog.shape}')
print(f'{'Sobel Edge':<15} {str(X_train_sobel.shape):<25} {X_test_sobel.shape}')

## Section 9: SVM Experiments — Hyperparameter Sweep

### About SVM
Support Vector Machine (SVM) is a classifier that finds the optimal boundary (hyperplane) that separates different classes with the maximum margin.

### Key Hyperparameter — C
The `C` parameter controls the trade-off between having a smooth decision boundary and classifying all training points correctly:
- **Small C** — wider margin, allows some misclassification, reduces overfitting, better generalisation
- **Large C** — narrower margin, tries to classify all training points correctly, may overfit

### Kernel Choice
- I use `linear` kernel for **Raw Pixels** and **Sobel** because they are high-dimensional (2025 features). Linear kernels are more efficient and work better in high-dimensional spaces.
- I use `rbf` kernel for **LBP** and **HOG** because they are lower-dimensional and the `rbf` kernel works well with compact feature representations.

### Experiment Setup
I use the same `run_svm_experiment()` function for all SVM runs. Each call creates a separate ClearML training task and logs all hyperparameters and results.

| Feature | Kernel | C values tested | Reason |
|---|---|---|---|
| Raw Pixels | linear | 0.01, 0.1, 1.0 | High-dimensional (2025) — smaller C reduces overfitting |
| LBP | rbf | 1.0, 10.0, 100.0 | Low-dimensional histogram — rbf kernel suits compact features |
| HOG | rbf | 10.0, 50.0, 100.0 | Test if higher C improves accuracy on HOG features |
| Sobel Edge | linear | 0.01, 0.1, 1.0 | High-dimensional (2025) — same reasoning as Raw Pixels |

In [ ]:
svm_results = {}

def run_svm_experiment(X_train_feat, X_test_feat, y_train, y_test,
                        feature_name, kernel='rbf', C=100.0, gamma='scale'):
    task_name = f'SVM | {feature_name} | kernel={kernel} | C={C}'
    print(f'\n{"="*60}')
    print(f'Running: {task_name}')
    print(f'Train samples: {len(X_train_feat)}, Test samples: {len(X_test_feat)}')
    print(f'Feature dim  : {X_train_feat.shape[1]}')

    svm_task = Task.init(
        project_name=CLEARML_PROJECT,
        task_name=task_name,
        task_type=Task.TaskTypes.training,
        reuse_last_task_id=False
    )
    svm_task.connect({
        'classifier'   : 'SVM',
        'feature'      : feature_name,
        'kernel'       : kernel,
        'C'            : C,
        'gamma'        : gamma,
        'random_state' : RANDOM_SEED,
        'n_train'      : len(X_train_feat),
        'n_test'       : len(X_test_feat),
        'feature_dim'  : X_train_feat.shape[1],
    })
    svm_logger = svm_task.get_logger()

    svm_model = svm.SVC(kernel=kernel, C=C, gamma=gamma, random_state=RANDOM_SEED)
    svm_model.fit(X_train_feat, y_train)

    train_acc = svm_model.score(X_train_feat, y_train)
    y_pred    = svm_model.predict(X_test_feat)
    test_acc  = accuracy_score(y_test, y_pred)

    print(f'Train accuracy : {train_acc:.4f}')
    print(f'Test accuracy  : {test_acc:.4f}')
    print(classification_report(y_test, y_pred,
                                  target_names=[idx_to_label[i] for i in range(NUM_CLASSES)]))

    svm_logger.report_scalar('Accuracy', 'Train', value=train_acc, iteration=0)
    svm_logger.report_scalar('Accuracy', 'Test',  value=test_acc,  iteration=0)

    cm = confusion_matrix(y_test, y_pred)
    class_labels = [idx_to_label[i] for i in range(NUM_CLASSES)]
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_labels, yticklabels=class_labels, ax=ax)
    ax.set_title(f'Confusion Matrix — {task_name}\nTest Accuracy: {test_acc:.4f}', fontsize=12)
    ax.set_ylabel('Actual Label')
    ax.set_xlabel('Predicted Label')
    plt.tight_layout()
    cm_path = f'/tmp/cm_svm_{feature_name.lower().replace(" ","_")}_C{C}.png'
    plt.savefig(cm_path, dpi=120)
    plt.show()
    plt.close(fig)

    svm_logger.report_image(title='Confusion Matrix', series=task_name,
                             local_path=cm_path, iteration=0)
    svm_logger.flush()
    svm_task.close()
    print(f'ClearML task closed ✅')

    return test_acc, y_pred, svm_model


print('SVM experiment runner ready ✅')

### Section 9.1: SVM + Raw Pixels — Varying C

I train the SVM with raw pixel features (2025-dimensional) using three values of `C`: 0.01, 0.1, and 1.0.

**Why these C values?**  
Raw pixel features have 2025 dimensions. With high-dimensional data, the SVM can easily memorise the training data, leading to a perfect training accuracy but lower test accuracy (overfitting). I test smaller values of `C` to add more regularisation and improve the test accuracy.

In [ ]:
svm_raw_c001_acc, svm_raw_c001_pred, _ = run_svm_experiment(
    X_train_raw, X_test_raw, y_train, y_test,
    feature_name='Raw-Pixels', kernel='linear', C=0.01
)
svm_results['SVM + Raw | C=0.01'] = svm_raw_c001_acc

svm_raw_c01_acc, svm_raw_c01_pred, _ = run_svm_experiment(
    X_train_raw, X_test_raw, y_train, y_test,
    feature_name='Raw-Pixels', kernel='linear', C=0.1
)
svm_results['SVM + Raw | C=0.1'] = svm_raw_c01_acc

svm_raw_c1_acc, svm_raw_c1_pred, _ = run_svm_experiment(
    X_train_raw, X_test_raw, y_train, y_test,
    feature_name='Raw-Pixels', kernel='linear', C=1.0
)
svm_results['SVM + Raw | C=1.0'] = svm_raw_c1_acc

print(f'\nRaw Pixels C sweep summary:')
print(f'  C=0.01 : {svm_raw_c001_acc:.4f}')
print(f'  C=0.1  : {svm_raw_c01_acc:.4f}')
print(f'  C=1.0  : {svm_raw_c1_acc:.4f}')

### Section 9.2: SVM + LBP — Varying C

I train the SVM with LBP features (26-dimensional histogram) using three values of `C`: 1.0, 10.0, and 100.0.

**Why these C values?**  
LBP produces a compact 26-dimensional feature vector. With low-dimensional features, the model is less likely to overfit, so I test larger values of `C` to allow the SVM to fit the data more aggressively.

LBP captures texture patterns which may not be the most discriminative feature for symbol classification — the accuracy may be lower than HOG because symbols differ more in shape than in texture.

In [ ]:
svm_lbp_c1_acc, svm_lbp_c1_pred, _ = run_svm_experiment(
    X_train_lbp, X_test_lbp, y_train, y_test,
    feature_name='LBP', kernel='rbf', C=1.0
)
svm_results['SVM + LBP | C=1.0'] = svm_lbp_c1_acc

svm_lbp_c10_acc, svm_lbp_c10_pred, _ = run_svm_experiment(
    X_train_lbp, X_test_lbp, y_train, y_test,
    feature_name='LBP', kernel='rbf', C=10.0
)
svm_results['SVM + LBP | C=10.0'] = svm_lbp_c10_acc

svm_lbp_c100_acc, svm_lbp_c100_pred, _ = run_svm_experiment(
    X_train_lbp, X_test_lbp, y_train, y_test,
    feature_name='LBP', kernel='rbf', C=100.0
)
svm_results['SVM + LBP | C=100.0'] = svm_lbp_c100_acc

print(f'\nLBP C sweep summary:')
print(f'  C=1.0  : {svm_lbp_c1_acc:.4f}')
print(f'  C=10.0 : {svm_lbp_c10_acc:.4f}')
print(f'  C=100.0: {svm_lbp_c100_acc:.4f}')

### Section 9.3: SVM + HOG — Varying C

I train the SVM with HOG features using three values of `C`: 10.0, 50.0, and 100.0.

**Why these C values?**  
HOG captures edge orientation, which is very relevant for distinguishing handwritten symbols that have different stroke patterns and shapes. I expect HOG to give the best accuracy among all features because symbols like `H`, `]`, `8`, and `alpha` differ significantly in their edge structure.

I test `C=10`, `C=50`, and `C=100` to find the best balance between fitting the training data and generalising to the test set.

In [ ]:
svm_hog_c10_acc, svm_hog_c10_pred, _ = run_svm_experiment(
    X_train_hog, X_test_hog, y_train, y_test,
    feature_name='HOG', kernel='rbf', C=10.0
)
svm_results['SVM + HOG | C=10.0'] = svm_hog_c10_acc

svm_hog_c50_acc, svm_hog_c50_pred, _ = run_svm_experiment(
    X_train_hog, X_test_hog, y_train, y_test,
    feature_name='HOG', kernel='rbf', C=50.0
)
svm_results['SVM + HOG | C=50.0'] = svm_hog_c50_acc

svm_hog_c100_acc, svm_hog_c100_pred, _ = run_svm_experiment(
    X_train_hog, X_test_hog, y_train, y_test,
    feature_name='HOG', kernel='rbf', C=100.0
)
svm_results['SVM + HOG | C=100.0'] = svm_hog_c100_acc

print(f'\nHOG C sweep summary:')
print(f'  C=10.0 : {svm_hog_c10_acc:.4f}')
print(f'  C=50.0 : {svm_hog_c50_acc:.4f}')
print(f'  C=100.0: {svm_hog_c100_acc:.4f}')

### Section 9.4: SVM + Sobel Edge — Varying C

I train the SVM with Sobel edge features (2025-dimensional) using three values of `C`: 0.01, 0.1, and 1.0.

**Why these C values?**  
Like raw pixels, Sobel features are high-dimensional (2025 values after flattening the edge map). I use smaller values of `C` to reduce the risk of overfitting. I expect Sobel features to perform better than raw pixels because they focus on the edges and strokes of each symbol rather than the background pixel intensities.

In [ ]:
svm_sobel_c001_acc, svm_sobel_c001_pred, _ = run_svm_experiment(
    X_train_sobel, X_test_sobel, y_train, y_test,
    feature_name='Sobel-Edge', kernel='linear', C=0.01
)
svm_results['SVM + Sobel | C=0.01'] = svm_sobel_c001_acc

svm_sobel_c01_acc, svm_sobel_c01_pred, _ = run_svm_experiment(
    X_train_sobel, X_test_sobel, y_train, y_test,
    feature_name='Sobel-Edge', kernel='linear', C=0.1
)
svm_results['SVM + Sobel | C=0.1'] = svm_sobel_c01_acc

svm_sobel_c1_acc, svm_sobel_c1_pred, _ = run_svm_experiment(
    X_train_sobel, X_test_sobel, y_train, y_test,
    feature_name='Sobel-Edge', kernel='linear', C=1.0
)
svm_results['SVM + Sobel | C=1.0'] = svm_sobel_c1_acc

print(f'\nSobel C sweep summary:')
print(f'  C=0.01 : {svm_sobel_c001_acc:.4f}')
print(f'  C=0.1  : {svm_sobel_c01_acc:.4f}')
print(f'  C=1.0  : {svm_sobel_c1_acc:.4f}')

## Section 10: ANN Experiments — Hyperparameter Sweep

### About ANN
An Artificial Neural Network (ANN) learns a mapping from input features to class labels through multiple layers of neurons. Each layer applies a weighted transformation followed by a non-linear activation function.

**Important:** I use **Dense (fully connected) layers only — no CNN layers**, as required by the assignment specification.

### ANN Architecture
All models follow the same structure:
```
Input → Dense(hidden_1, relu) → Dense(hidden_2, relu) → Dense(10, softmax)
```
The output layer has 10 neurons — one for each class — with `softmax` activation to produce class probabilities.

### Key Hyperparameter — Hidden Layer Size
The number of neurons in the hidden layers controls the capacity of the network:
- **Too few neurons** — the model may underfit and not learn complex patterns
- **Too many neurons** — the model may overfit the training data

I test three different sizes to find the right balance for each feature type.

### Fixed Settings (same for all ANN experiments)
| Setting | Value |
|---|---|
| Activation (hidden) | relu |
| Activation (output) | softmax |
| Loss function | sparse_categorical_crossentropy |
| Optimiser | Adam |
| Learning rate | 0.001 |
| Epochs | 50 |
| Batch size | 32 |
| Validation split | 10% |

### Architectures Tested
| Architecture | Hidden Layer 1 | Hidden Layer 2 | Description |
|---|---|---|---|
| Small | 128 | 64 | Simpler model — fewer parameters |
| Medium | 256 | 128 | Standard model |
| Large | 512 | 256 | More complex model — higher capacity |

In [ ]:
ANN_EPOCHS           = 50
ANN_BATCH_SIZE       = 32
ANN_LEARNING_RATE    = 0.001
ANN_VALIDATION_SPLIT = 0.10

ann_results = {}

def run_ann_experiment(X_train_feat, X_test_feat, y_train, y_test,
                        feature_name, hidden_1=256, hidden_2=128):
    n_features = X_train_feat.shape[1]
    task_name  = f'ANN | {feature_name} | Dense({hidden_1},{hidden_2},10)'

    print(f'\n{"="*60}')
    print(f'Running: {task_name}')
    print(f'Feature dim: {n_features}, Train: {len(X_train_feat)}, Test: {len(X_test_feat)}')

    ann_task = Task.init(
        project_name=CLEARML_PROJECT,
        task_name=task_name,
        task_type=Task.TaskTypes.training,
        reuse_last_task_id=False
    )
    ann_task.connect({
        'classifier'        : 'ANN',
        'feature'           : feature_name,
        'input_dim'         : n_features,
        'hidden_layer_1'    : hidden_1,
        'hidden_layer_2'    : hidden_2,
        'output_neurons'    : NUM_CLASSES,
        'activation_hidden' : 'relu',
        'activation_output' : 'softmax',
        'loss_function'     : 'sparse_categorical_crossentropy',
        'optimizer'         : 'Adam',
        'learning_rate'     : ANN_LEARNING_RATE,
        'epochs'            : ANN_EPOCHS,
        'batch_size'        : ANN_BATCH_SIZE,
        'validation_split'  : ANN_VALIDATION_SPLIT,
        'n_train'           : len(X_train_feat),
        'n_test'            : len(X_test_feat),
    })
    ann_logger = ann_task.get_logger()

    ann_model = tf.keras.models.Sequential([
        tf.keras.layers.Dense(hidden_1, activation=tf.nn.relu,
                               input_shape=[n_features],
                               name='hidden_layer_1'),
        tf.keras.layers.Dense(hidden_2, activation=tf.nn.relu,
                               name='hidden_layer_2'),
        tf.keras.layers.Dense(NUM_CLASSES, activation=tf.nn.softmax,
                               name='output_layer')
    ])

    ann_model.compile(
        optimizer=tf.optimizers.Adam(learning_rate=ANN_LEARNING_RATE),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    ann_model.summary()

    history = ann_model.fit(
        X_train_feat, y_train,
        epochs=ANN_EPOCHS,
        batch_size=ANN_BATCH_SIZE,
        validation_split=ANN_VALIDATION_SPLIT,
        verbose=1
    )

    import pandas as pd
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history.history['accuracy'],     label='Train Accuracy')
    axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
    axes[0].set_title(f'ANN Accuracy — {feature_name} Dense({hidden_1},{hidden_2})')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True)
    axes[1].plot(history.history['loss'],     label='Train Loss')
    axes[1].plot(history.history['val_loss'], label='Val Loss')
    axes[1].set_title(f'ANN Loss — {feature_name} Dense({hidden_1},{hidden_2})')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True)
    plt.tight_layout()
    curve_path = f'/tmp/ann_curves_{feature_name.lower().replace(" ","_")}_{hidden_1}_{hidden_2}.png'
    plt.savefig(curve_path, dpi=120)
    plt.show()
    plt.close(fig)
    ann_logger.report_image(title='Training Curves', series=task_name,
                             local_path=curve_path, iteration=0)

    test_loss, test_acc  = ann_model.evaluate(X_test_feat, y_test, verbose=0)
    train_acc_final      = history.history['accuracy'][-1]
    y_pred               = np.argmax(ann_model.predict(X_test_feat), axis=1)

    print(f'\nFinal Train accuracy : {train_acc_final:.4f}')
    print(f'Test accuracy        : {test_acc:.4f}')
    print(classification_report(y_test, y_pred,
                                  target_names=[idx_to_label[i] for i in range(NUM_CLASSES)]))

    ann_logger.report_scalar('Accuracy', 'Train', value=train_acc_final, iteration=0)
    ann_logger.report_scalar('Accuracy', 'Test',  value=test_acc,        iteration=0)
    for epoch_idx, (tr_acc, val_acc) in enumerate(
            zip(history.history['accuracy'], history.history['val_accuracy'])):
        ann_logger.report_scalar('Epoch Accuracy', 'Train', value=tr_acc,  iteration=epoch_idx)
        ann_logger.report_scalar('Epoch Accuracy', 'Val',   value=val_acc, iteration=epoch_idx)

    cm = confusion_matrix(y_test, y_pred)
    class_labels = [idx_to_label[i] for i in range(NUM_CLASSES)]
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_labels, yticklabels=class_labels, ax=ax)
    ax.set_title(f'Confusion Matrix — {task_name}\nTest Accuracy: {test_acc:.4f}', fontsize=12)
    ax.set_ylabel('Actual Label')
    ax.set_xlabel('Predicted Label')
    plt.tight_layout()
    cm_path = f'/tmp/cm_ann_{feature_name.lower().replace(" ","_")}_{hidden_1}_{hidden_2}.png'
    plt.savefig(cm_path, dpi=120)
    plt.show()
    plt.close(fig)
    ann_logger.report_image(title='Confusion Matrix', series=task_name,
                             local_path=cm_path, iteration=0)
    ann_logger.flush()
    ann_task.close()
    print(f'ClearML task closed ✅')

    return test_acc, y_pred, ann_model


print('ANN experiment runner ready ✅')

### Section 10.1: ANN + Raw Pixels — Varying Hidden Layer Size

I train the ANN using raw pixel features (2025-dimensional input) with three different hidden layer sizes.

**Why vary the layer size?**  
With 2025 input features, the network receives a lot of information. A larger network may be able to learn more complex patterns from the raw pixels, but may also overfit. I test three sizes to find the best trade-off.

In [ ]:
ann_raw_pixels_s_acc, ann_raw_pixels_s_pred, _ = run_ann_experiment(
    X_train_raw, X_test_raw, y_train, y_test,
    feature_name='Raw-Pixels', hidden_1=128, hidden_2=64
)
ann_results['ANN + Raw-Pixels | Dense(128,64)']  = ann_raw_pixels_s_acc

ann_raw_pixels_m_acc, ann_raw_pixels_m_pred, _ = run_ann_experiment(
    X_train_raw, X_test_raw, y_train, y_test,
    feature_name='Raw-Pixels', hidden_1=256, hidden_2=128
)
ann_results['ANN + Raw-Pixels | Dense(256,128)'] = ann_raw_pixels_m_acc

ann_raw_pixels_l_acc, ann_raw_pixels_l_pred, _ = run_ann_experiment(
    X_train_raw, X_test_raw, y_train, y_test,
    feature_name='Raw-Pixels', hidden_1=512, hidden_2=256
)
ann_results['ANN + Raw-Pixels | Dense(512,256)'] = ann_raw_pixels_l_acc

print(f'\nRaw-Pixels ANN architecture sweep:')
print(f'  Dense(128,64)  : {ann_raw_pixels_s_acc:.4f}')
print(f'  Dense(256,128) : {ann_raw_pixels_m_acc:.4f}')
print(f'  Dense(512,256) : {ann_raw_pixels_l_acc:.4f}')

### Section 10.2: ANN + LBP — Varying Hidden Layer Size

I train the ANN using LBP features (26-dimensional input) with three different hidden layer sizes.

**Why vary the layer size?**  
LBP features are very compact (only 26 values). With such a low-dimensional input, a very large network may not add much benefit and could overfit. I test whether a smaller architecture works just as well or better for this feature.

In [ ]:
ann_lbp_s_acc, ann_lbp_s_pred, _ = run_ann_experiment(
    X_train_lbp, X_test_lbp, y_train, y_test,
    feature_name='LBP', hidden_1=128, hidden_2=64
)
ann_results['ANN + LBP | Dense(128,64)']  = ann_lbp_s_acc

ann_lbp_m_acc, ann_lbp_m_pred, _ = run_ann_experiment(
    X_train_lbp, X_test_lbp, y_train, y_test,
    feature_name='LBP', hidden_1=256, hidden_2=128
)
ann_results['ANN + LBP | Dense(256,128)'] = ann_lbp_m_acc

ann_lbp_l_acc, ann_lbp_l_pred, _ = run_ann_experiment(
    X_train_lbp, X_test_lbp, y_train, y_test,
    feature_name='LBP', hidden_1=512, hidden_2=256
)
ann_results['ANN + LBP | Dense(512,256)'] = ann_lbp_l_acc

print(f'\nLBP ANN architecture sweep:')
print(f'  Dense(128,64)  : {ann_lbp_s_acc:.4f}')
print(f'  Dense(256,128) : {ann_lbp_m_acc:.4f}')
print(f'  Dense(512,256) : {ann_lbp_l_acc:.4f}')

### Section 10.3: ANN + HOG — Varying Hidden Layer Size

I train the ANN using HOG features with three different hidden layer sizes.

**Why vary the layer size?**  
HOG features capture edge direction information which is highly relevant for distinguishing handwritten symbols. I want to see whether a larger network can extract even more useful patterns from the HOG representation.

In [ ]:
ann_hog_s_acc, ann_hog_s_pred, _ = run_ann_experiment(
    X_train_hog, X_test_hog, y_train, y_test,
    feature_name='HOG', hidden_1=128, hidden_2=64
)
ann_results['ANN + HOG | Dense(128,64)']  = ann_hog_s_acc

ann_hog_m_acc, ann_hog_m_pred, _ = run_ann_experiment(
    X_train_hog, X_test_hog, y_train, y_test,
    feature_name='HOG', hidden_1=256, hidden_2=128
)
ann_results['ANN + HOG | Dense(256,128)'] = ann_hog_m_acc

ann_hog_l_acc, ann_hog_l_pred, _ = run_ann_experiment(
    X_train_hog, X_test_hog, y_train, y_test,
    feature_name='HOG', hidden_1=512, hidden_2=256
)
ann_results['ANN + HOG | Dense(512,256)'] = ann_hog_l_acc

print(f'\nHOG ANN architecture sweep:')
print(f'  Dense(128,64)  : {ann_hog_s_acc:.4f}')
print(f'  Dense(256,128) : {ann_hog_m_acc:.4f}')
print(f'  Dense(512,256) : {ann_hog_l_acc:.4f}')

### Section 10.4: ANN + Sobel Edge — Varying Hidden Layer Size

I train the ANN using Sobel edge features (2025-dimensional input) with three different hidden layer sizes.

**Why vary the layer size?**  
Sobel features highlight the edges and strokes of each symbol. Since symbols are largely defined by their edges, I expect the ANN to perform reasonably well on this feature. I test three layer sizes to see which architecture best captures the edge patterns.

In [ ]:
ann_sobel_edge_s_acc, ann_sobel_edge_s_pred, _ = run_ann_experiment(
    X_train_sobel, X_test_sobel, y_train, y_test,
    feature_name='Sobel-Edge', hidden_1=128, hidden_2=64
)
ann_results['ANN + Sobel-Edge | Dense(128,64)']  = ann_sobel_edge_s_acc

ann_sobel_edge_m_acc, ann_sobel_edge_m_pred, _ = run_ann_experiment(
    X_train_sobel, X_test_sobel, y_train, y_test,
    feature_name='Sobel-Edge', hidden_1=256, hidden_2=128
)
ann_results['ANN + Sobel-Edge | Dense(256,128)'] = ann_sobel_edge_m_acc

ann_sobel_edge_l_acc, ann_sobel_edge_l_pred, _ = run_ann_experiment(
    X_train_sobel, X_test_sobel, y_train, y_test,
    feature_name='Sobel-Edge', hidden_1=512, hidden_2=256
)
ann_results['ANN + Sobel-Edge | Dense(512,256)'] = ann_sobel_edge_l_acc

print(f'\nSobel-Edge ANN architecture sweep:')
print(f'  Dense(128,64)  : {ann_sobel_edge_s_acc:.4f}')
print(f'  Dense(256,128) : {ann_sobel_edge_m_acc:.4f}')
print(f'  Dense(512,256) : {ann_sobel_edge_l_acc:.4f}')

## Section 11: Results — Comparative Study

In this section, I compare all 24 experiment results — 12 SVM runs and 12 ANN runs.

The following results are presented:
1. **Full ranked results table** — all 24 experiments sorted from best to worst test accuracy
2. **Bar chart comparison** — SVM and ANN results shown side by side for easy visual comparison
3. **Confusion matrix of the best combination** — shows which classes are confused with each other
4. **Sample correct and incorrect predictions** — images classified correctly (green) and incorrectly (red), with true label (T) and predicted label (P)

The confusion matrix and sample predictions help in understanding **why** certain classifier and feature combinations perform better or worse than others.

In [ ]:
all_results    = {**svm_results, **ann_results}
sorted_results = sorted(all_results.items(), key=lambda x: x[1], reverse=True)

print('\n' + '='*65)
print('HYPERPARAMETER SWEEP — FULL RESULTS')
print('='*65)
print(f'{'Experiment':<40} {'Test Accuracy':>15}  Rank')
print('-'*65)
for rank, (name, acc) in enumerate(sorted_results, 1):
    marker = ' ← BEST' if rank == 1 else ''
    print(f'{name:<40} {acc:>15.4f}  #{rank}{marker}')
print('='*65)

best_name = sorted_results[0][0]
best_acc  = sorted_results[0][1]
print(f'\nBEST CONFIGURATION : {best_name}')
print(f'BEST TEST ACCURACY : {best_acc:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

svm_sorted = sorted(svm_results.items(), key=lambda x: x[1], reverse=True)
ann_sorted = sorted(ann_results.items(), key=lambda x: x[1], reverse=True)

svm_names  = [n for n, _ in svm_sorted]
svm_accs   = [a for _, a in svm_sorted]
svm_colors = ['#1565C0' if a == max(svm_accs) else '#90CAF9' for a in svm_accs]
bars = axes[0].barh(svm_names, svm_accs, color=svm_colors, edgecolor='black')
axes[0].bar_label(bars, fmt='%.4f', padding=3, fontsize=9, fontweight='bold')
axes[0].set_xlim(0, 1.15)
axes[0].set_title('SVM — All Hyperparameter Sweep Results', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Test Accuracy')
axes[0].grid(axis='x', alpha=0.3)

ann_names  = [n for n, _ in ann_sorted]
ann_accs   = [a for _, a in ann_sorted]
ann_colors = ['#2E7D32' if a == max(ann_accs) else '#A5D6A7' for a in ann_accs]
bars = axes[1].barh(ann_names, ann_accs, color=ann_colors, edgecolor='black')
axes[1].bar_label(bars, fmt='%.4f', padding=3, fontsize=9, fontweight='bold')
axes[1].set_xlim(0, 1.15)
axes[1].set_title('ANN — All Hyperparameter Sweep Results', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Test Accuracy')
axes[1].grid(axis='x', alpha=0.3)

fig.suptitle('Assignment 1 — Hyperparameter Sweep Results (Student: 14679308)',
              fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/sweep_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sweep results chart saved ✅')

In [ ]:
print(f'Best combination: {best_name}  (accuracy = {best_acc:.4f})')

best_pred_map = {
    'SVM + Raw | C=0.01'           : svm_raw_c001_pred,
    'SVM + Raw | C=0.1'            : svm_raw_c01_pred,
    'SVM + Raw | C=1.0'            : svm_raw_c1_pred,
    'SVM + LBP | C=1.0'            : svm_lbp_c1_pred,
    'SVM + LBP | C=10.0'           : svm_lbp_c10_pred,
    'SVM + LBP | C=100.0'          : svm_lbp_c100_pred,
    'SVM + HOG | C=10.0'           : svm_hog_c10_pred,
    'SVM + HOG | C=50.0'           : svm_hog_c50_pred,
    'SVM + HOG | C=100.0'          : svm_hog_c100_pred,
    'SVM + Sobel | C=0.01'         : svm_sobel_c001_pred,
    'SVM + Sobel | C=0.1'          : svm_sobel_c01_pred,
    'SVM + Sobel | C=1.0'          : svm_sobel_c1_pred,
    'ANN + Raw-Pixels | Dense(128,64)'  : ann_raw_pixels_s_pred,
    'ANN + Raw-Pixels | Dense(256,128)' : ann_raw_pixels_m_pred,
    'ANN + Raw-Pixels | Dense(512,256)' : ann_raw_pixels_l_pred,
    'ANN + LBP | Dense(128,64)'    : ann_lbp_s_pred,
    'ANN + LBP | Dense(256,128)'   : ann_lbp_m_pred,
    'ANN + LBP | Dense(512,256)'   : ann_lbp_l_pred,
    'ANN + HOG | Dense(128,64)'    : ann_hog_s_pred,
    'ANN + HOG | Dense(256,128)'   : ann_hog_m_pred,
    'ANN + HOG | Dense(512,256)'   : ann_hog_l_pred,
    'ANN + Sobel-Edge | Dense(128,64)'  : ann_sobel_edge_s_pred,
    'ANN + Sobel-Edge | Dense(256,128)' : ann_sobel_edge_m_pred,
    'ANN + Sobel-Edge | Dense(512,256)' : ann_sobel_edge_l_pred,
}

best_y_pred  = best_pred_map[best_name]
class_labels = [idx_to_label[i] for i in range(NUM_CLASSES)]

cm_best = confusion_matrix(y_test, best_y_pred)
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm_best, annot=True, fmt='d', cmap='Blues',
             xticklabels=class_labels, yticklabels=class_labels,
             linewidths=0.5, square=True, ax=ax)
ax.set_title(f'Confusion Matrix — BEST: {best_name}\nTest Accuracy: {best_acc:.4f}',
              fontsize=12, fontweight='bold')
ax.set_ylabel('Actual Label', fontsize=12)
ax.set_xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('/tmp/best_confusion_matrix_sweep.png', dpi=150)
plt.show()
print('Best confusion matrix saved ✅')

In [ ]:
correct_idx   = np.where(best_y_pred == y_test)[0]
incorrect_idx = np.where(best_y_pred != y_test)[0]

np.random.seed(RANDOM_SEED)
n_show           = 8
correct_sample   = np.random.choice(correct_idx,   min(n_show, len(correct_idx)),   replace=False)
incorrect_sample = np.random.choice(incorrect_idx, min(n_show, len(incorrect_idx)), replace=False)

fig, axes = plt.subplots(2, n_show, figsize=(n_show * 2, 5))
fig.suptitle(f'Sample Predictions — {best_name}\nTop row: Correct | Bottom row: Incorrect',
              fontsize=12, fontweight='bold')

for col, idx in enumerate(correct_sample):
    ax  = axes[0][col]
    img = X_test_imgs[idx].astype(np.uint8)
    ax.imshow(img, cmap='gray')
    ax.axis('off')
    ax.set_title(f'T:{idx_to_label[y_test[idx]]}\nP:{idx_to_label[best_y_pred[idx]]}',
                  fontsize=7, color='green')

for col, idx in enumerate(incorrect_sample):
    ax  = axes[1][col]
    img = X_test_imgs[idx].astype(np.uint8)
    ax.imshow(img, cmap='gray')
    ax.axis('off')
    ax.set_title(f'T:{idx_to_label[y_test[idx]]}\nP:{idx_to_label[best_y_pred[idx]]}',
                  fontsize=7, color='red')

plt.tight_layout()
plt.savefig('/tmp/sample_predictions_sweep.png', dpi=150)
plt.show()
print('Sample predictions saved ✅')
print('T = True label  |  P = Predicted label')
print('Green = Correct  |  Red = Incorrect')

## Section 12: ClearML — Results Summary Task

In this final section, I create a ClearML summary task that logs the overall results of the hyperparameter sweep.

The following are logged to ClearML:
- Test accuracy for all 24 experiments as scalar values
- The name and accuracy of the best performing configuration
- The sweep comparison bar chart
- The confusion matrix of the best combination
- Sample correct and incorrect predictions

This summary task provides a single place in ClearML where all the results of this notebook can be viewed at a glance.

In [ ]:
summary_task = Task.init(
    project_name=CLEARML_PROJECT,
    task_name='Results-Summary-Hyperparameter-Sweep',
    task_type=Task.TaskTypes.testing,
    reuse_last_task_id=False
)

summary_logger = summary_task.get_logger()

for exp_name, acc in all_results.items():
    summary_logger.report_scalar(title='Final Test Accuracy',
                                  series=exp_name, value=acc, iteration=0)

summary_task.connect({
    'best_combination' : best_name,
    'best_accuracy'    : best_acc,
})

summary_logger.report_image(title='Sweep Comparison Chart', series='SVM vs ANN',
                              local_path='/tmp/sweep_results.png', iteration=0)
summary_logger.report_image(title='Best Confusion Matrix', series=best_name,
                              local_path='/tmp/best_confusion_matrix_sweep.png', iteration=0)
summary_logger.report_image(title='Sample Predictions', series='correct vs incorrect',
                              local_path='/tmp/sample_predictions_sweep.png', iteration=0)

summary_logger.flush()
summary_task.close()

print('='*60)
print('HYPERPARAMETER SWEEP COMPLETE ✅')
print(f'Best configuration : {best_name}')
print(f'Best test accuracy : {best_acc:.4f}')
print(f'View on ClearML    : https://app.clear.ml')
print('='*60)